# Step 4: Latent Extraction

This notebook demonstrates how to extract visual latent features from aligned video frames using DINOv2 (or DINOv3).

In [ ]:
import numpy as np
from tqdm import tqdm
import cv2
import torch
import os

from castle.utils.visual_latent_extract import generate_dinov2
from castle.utils.video_io import ReadArray

import matplotlib.pyplot as plt

In [ ]:
# Initialize Model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# You can change this to 'dinov3_vitb16' if available
encoder = generate_dinov2(model_type='dinov2_vitb14_reg', device=device)

In [ ]:
# Paths
video_align_path = 'temp/video-align.mp4'
mask_video_align_path = 'temp/mask-align.mp4'
video_latent_path = 'temp/video_latent.npz'

# Ensure temp dir exists
os.makedirs('temp', exist_ok=True)

# Read Video Data
# Note: ReadArray reads the entire video into memory. For large videos, verify memory limits.
try:
    video_align = ReadArray(video_align_path)
    mask_video_align = ReadArray(mask_video_align_path)
except FileNotFoundError:
    print("Video files not found in temp/. Please run previous steps or provide video files.")
    # Generate dummy data for demonstration if files missing
    print("Generating dummy data for demonstration...")
    video_align = np.random.randint(0, 255, (100, 224, 224, 3), dtype=np.uint8)
    mask_video_align = np.zeros((100, 224, 224, 3), dtype=np.uint8)
    mask_video_align[:, 50:150, 50:150] = [122, 228, 240] # Dummy mask

In [ ]:
# Configuration
roi_rgb = [122, 228, 240] # Target Mask Color (RGB)
BATCH_SIZE = 32

n = min(len(video_align), len(mask_video_align))
video_latents = []

frames_buffer = []
masks_buffer = []

def process_batch(frames, masks):
    """Process a batch of frames and RGB masks to extract latents."""
    int_masks = []
    tolerance = 30
    
    lower = np.array([c - tolerance for c in roi_rgb])
    upper = np.array([c + tolerance for c in roi_rgb])

    for m in masks:
        # Create binary mask for the ROI color
        mask = cv2.inRange(m, lower, upper)
        # Convert to 0/1 integer mask (1 = ROI)
        int_masks.append((mask > 0).astype(np.uint8))
    
    # Extract latents (ROI ID = 1)
    latents = encoder.extract_batch_latent(frames, int_masks, select_roi=1)
    return latents

print(f"Processing {n} frames with batch size {BATCH_SIZE}...")
for i in tqdm(range(n)):
    frames_buffer.append(video_align[i])
    masks_buffer.append(mask_video_align[i])
    
    if len(frames_buffer) >= BATCH_SIZE:
        batch_latents = process_batch(frames_buffer, masks_buffer)
        video_latents.append(batch_latents)
        frames_buffer = []
        masks_buffer = []

# Process remaining frames
if frames_buffer:
    batch_latents = process_batch(frames_buffer, masks_buffer)
    video_latents.append(batch_latents)

# Concatenate and Save
if video_latents:
    all_latents = np.concatenate(video_latents, axis=0)
    np.savez_compressed(video_latent_path, latent=all_latents)
    print(f"Saved latents to {video_latent_path}")
    print(f"Shape: {all_latents.shape}")
else:
    print("No latents extracted.")